In [1]:
import json
import os
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
from typing import Literal, Optional, Union, List

import tiktoken
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field

env_path = Path("/home/luis/Documents/FGV/Laboratory/document-graph/server/.env")
load_dotenv(dotenv_path=env_path)

MODELLM = "gpt-4.1-mini"
APIKEY = os.getenv("OPENAI_API_KEY")

if not APIKEY:
    raise ValueError("OPENAI_API_KEY not found. Check your .env file.")

client = OpenAI(api_key=APIKEY)

INPUT_PATH = Path(
    "../../infra/json/graph/related_BELLICUMPHARMACEUTICALS_INC_05_07_2019-EX-10.1-Supply_Agreement.json"
)

OUTPUT_DIR = Path("../../infra/json/kg_extraction")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

KG_PARTS_PATH = OUTPUT_DIR / "bellicum_kg_parts.json"
KG_FULL_PATH = OUTPUT_DIR / "bellicum_contract_kg.json"
ERRORS_PATH = OUTPUT_DIR / "bellicum_kg_errors.json"

MAX_WORKERS = 1

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Input keys:", data.keys())
print("Total nodes:", len(data.get("nodes", [])))

Input keys: dict_keys(['documentId', 'nodes'])
Total nodes: 4


In [2]:
NodeType = Literal[
    "Clause",
    "DefinedTerm",
    "Party",
    "Obligation",
    "Right",
    "Permission",
    "Prohibition",
    "Condition",
    "Reference",
    "Value",
    "Time"
]

EdgeType = Literal[
    "CONTRADICTS"
]

ALLOWED_NODE_TYPES = set(NodeType.__args__)
ALLOWED_EDGE_TYPES = set(EdgeType.__args__)

In [3]:
class ClauseProperties(BaseModel):
    clause_id: str
    text: str
    title: Optional[str] = None
    paragraph_enum: Optional[int] = None
    level: Optional[int] = None


class DefinedTermProperties(BaseModel):
    term: str
    definition: Optional[str] = None
    definition_clause_id: Optional[str] = None


class PartyProperties(BaseModel):
    name: str
    role: Optional[str] = None
    address: Optional[str] = None
    canonical_name: Optional[str] = None


class ObligationProperties(BaseModel):
    actor: Optional[str] = None
    action: str
    normalized_action: Optional[str] = None
    object: Optional[str] = None
    normalized_object: Optional[str] = None
    deadline: Optional[str] = None
    condition: Optional[str] = None

class RightProperties(BaseModel):
    holder: Optional[str] = None
    action: str
    normalized_action: Optional[str] = None
    object: Optional[str] = None
    normalized_object: Optional[str] = None
    frequency: Optional[str] = None
    notice_period: Optional[str] = None
    condition: Optional[str] = None
    scope: Optional[str] = None
    exception: Optional[str] = None
    value_refs: List[str] = Field(default_factory=list)
    modality: Literal["right"] = "right"
    polarity: Literal["positive"] = "positive"


class PermissionProperties(BaseModel):
    holder: Optional[str] = None
    action: str
    normalized_action: Optional[str] = None
    object: Optional[str] = None
    normalized_object: Optional[str] = None
    condition: Optional[str] = None
    scope: Optional[str] = None
    exception: Optional[str] = None
    value_refs: List[str] = Field(default_factory=list)
    modality: Literal["permission"] = "permission"
    polarity: Literal["positive"] = "positive"


class ProhibitionProperties(BaseModel):
    subject: Optional[str] = None
    action: str
    normalized_action: Optional[str] = None
    object: Optional[str] = None
    normalized_object: Optional[str] = None
    condition: Optional[str] = None
    scope: Optional[str] = None
    exception: Optional[str] = None
    value_refs: List[str] = Field(default_factory=list)
    modality: Literal["prohibition"] = "prohibition"
    polarity: Literal["negative"] = "negative"


class ConditionProperties(BaseModel):
    trigger: str
    subject: Optional[str] = None
    action: Optional[str] = None
    object: Optional[str] = None
    value: Optional[str] = None
    threshold: Optional[str] = None
    polarity: Literal["positive", "negative"] = "positive"


class ReferenceProperties(BaseModel):
    name: str
    citation: Optional[str] = None


class ValueProperties(BaseModel):
    amount: Optional[Union[str, int, float]] = None
    unit: Optional[str] = None
    normalized_value: Optional[str] = None
    applies_to: Optional[str] = None

In [4]:
class KGEntity(BaseModel):
    id: str
    type: NodeType
    label: str
    properties: dict
    evidence_text: str
    confidence: float = Field(ge=0.0, le=1.0)


class KGRelation(BaseModel):
    source: str
    target: str
    type: EdgeType
    confidence: float = Field(ge=0.0, le=1.0)
    evidence_text: str
    source_field: Optional[str] = None
    target_field: Optional[str] = None


class ClauseKGExtraction(BaseModel):
    clause_id: str
    entities: List[KGEntity]
    relations: List[KGRelation]

In [5]:
def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def safe_json_loads(text):
    text = text.strip()

    if text.startswith("```json"):
        text = text.removeprefix("```json").removesuffix("```").strip()
    elif text.startswith("```"):
        text = text.removeprefix("```").removesuffix("```").strip()

    start = text.find("{")
    end = text.rfind("}")

    if start == -1 or end == -1:
        raise ValueError("No JSON object found in model output.")

    return json.loads(text[start:end + 1])


def estimate_cost(input_tokens, output_tokens):
    input_cost = input_tokens * 0.0005 / 1000
    output_cost = output_tokens * 0.0015 / 1000
    return input_cost + output_cost

In [6]:
enc = tiktoken.encoding_for_model(MODELLM)

In [7]:
def build_clause_input(node):
    return {
        "clause_id": str(node.get("id")),
        "text": (node.get("text") or "").strip(),
        "title": node.get("title"),
        "paragraph_enum": node.get("paragraph_enum"),
        "level": node.get("level"),
    }

In [8]:
def make_extraction_prompt(nodes):
    return f"""
You are a expert finding contradictions and extracting a Contract Knowledge Graph from a legal contract clause.

Return only valid JSON. Do not return markdown.

Task:
Extract entities and relations according to the ontology.

Node types:
- DefinedTerm
- Party
- Obligation
- Right
- Permission
- Prohibition
- Condition
- Reference
- Value

Edge types:
- CONTRADICTS

Important extraction rules:
- Extract only what is explicitly supported by the clause text.
- Create exactly one Clause entity for the input clause.
- Every entity must include evidence_text.
- Every relation must include evidence_text.
- Relation source and target IDs must exist in entities.
- Use stable local IDs based on the clause_id.
- Do not invent missing information.
- If a property is unknown, use null.

Input clause:
{json.dumps(nodes, indent=2)}

Return exactly this JSON structure:

{{
  "entities": [
    {{
      "id": "string",
      "type": "Clause | DefinedTerm | Party | Obligation | Right | Permission | Prohibition | Condition | Reference | Value",
      "label": "string",
      "properties": {{}},
      "evidence_text": "string",
    }}
  ],
  "relations": [
    {{
      "source": "string",
      "target": "string",
      "type": "CONTRADICTS",
      "confidence": 0.0,
      "evidence_text": "string",
      "source_field": "string or null",
      "target_field": "string or null"
    }}
  ]
}}
"""

In [9]:
def extract_kg_from_clause(client, nodes):
    prompt = make_extraction_prompt(nodes)
    input_tokens = len(enc.encode(prompt))


    response = client.responses.create(
        model=MODELLM,
        input=prompt,
        temperature=0,
    )

    output_text = response.output_text
    output_tokens = len(enc.encode(output_text))
    cost = estimate_cost(input_tokens, output_tokens)

    kg_json = safe_json_loads(output_text)

    return kg_json, input_tokens, output_tokens, cost


In [10]:
# def validate_kg_part(kg_part):
#     if not kg_part:
#         return False, "Empty kg_part"

#     try:
#         parsed = ClauseKGExtraction.model_validate(kg_part)
#     except Exception as e:
#         return False, f"Pydantic validation failed: {e}"

#     entity_ids = {ent.id for ent in parsed.entities}

#     clause_entities = [ent for ent in parsed.entities if ent.type == "Clause"]
#     if len(clause_entities) != 1:
#         return False, f"Expected exactly one Clause entity, found {len(clause_entities)}"

#     for ent in parsed.entities:
#         if not ent.id:
#             return False, "Entity without id"

#         if not ent.label:
#             return False, f"Entity {ent.id} without label"

#         if not ent.evidence_text:
#             return False, f"Entity {ent.id} without evidence_text"

#     for rel in parsed.relations:
#         if rel.source not in entity_ids:
#             return False, f"Relation source does not exist: {rel.source}"

#         if rel.target not in entity_ids:
#             return False, f"Relation target does not exist: {rel.target}"

#         if not rel.evidence_text:
#             return False, f"Relation {rel.source}->{rel.target} without evidence_text"

#     return True, None

In [11]:
nodes = [
    node for node in data["nodes"]
    if len((node.get("text") or "").strip()) >= 30
]

print("Nodes to process:", len(nodes))

kg_json, input_tokens, output_tokens, cost = extract_kg_from_clause(client, nodes)
break

Nodes to process: 4


SyntaxError: 'break' outside loop (1852388473.py, line 9)

In [12]:
kg_json

{'entities': [{'id': 'clause_related::BELLICUMPHARMACEUTICALS_INC_05_07_2019-EX-10.1-Supply_Agreement-p-216',
   'type': 'Clause',
   'label': 'Clause related::BELLICUMPHARMACEUTICALS_INC_05_07_2019-EX-10.1-Supply_Agreement-p-216',
   'properties': {},
   'evidence_text': 'Price necessitated by any such continued supply of unchanged Miltenyi Product during such period. Until such agreement is reached, any limitations on or obligations of Bellicum under Article 5 pertaining to forecast variances and Firm Zone ordering in relation to Miltenyi Products described in this subsection (g) shall not apply, and therefore Bellicum has no obligation to purchase any such Miltenyi Products produced after implementation of such Material Change. If the continued supply of unchanged Miltenyi Product under this subsection (g) is reasonably estimated by the Parties to exceed a period of six (6) months from the implementation date of the Material Change notified in a Change Notification pursuant to Secti

In [ ]:
# def process_node(i, node):
#     clause = build_clause_input(node)

#     try:
#         # kg_part, in_t, out_t, cost = extract_kg_from_clause(client, clause)

#         # valid, validation_error = validate_kg_part(kg_part)

#         return {
#             "ok": valid,
#             "index": i,
#             "clause": clause,
#             "kg_part": kg_part,
#             "input_tokens": in_t,
#             "output_tokens": out_t,
#             "cost": cost,
#             "error": validation_error,
#         }

#     except Exception as e:
#         return {
#             "ok": False,
#             "index": i,
#             "clause": clause,
#             "kg_part": None,
#             "input_tokens": 0,
#             "output_tokens": 0,
#             "cost": 0,
#             "error": str(e),
#         }

In [ ]:
kg_parts = []
errors = []

total_tokens = 0
total_cost = 0.0

nodes

NameError: name 'nodes' is not defined

In [ ]:
save_json(kg_parts, KG_PARTS_PATH)
save_json(errors, ERRORS_PATH)

print("Done.")
print("Valid KG parts:", len(kg_parts))
print("Errors:", len(errors))
print("Total tokens:", total_tokens)
print(f"Total estimated cost: ${total_cost:.4f}")

Done.
Valid KG parts: 2
Errors: 0
Total tokens: 3221
Total estimated cost: $0.0036


In [ ]:
def merge_evidence(old_ev, new_ev):
    if old_ev is None:
        old_list = []
    elif isinstance(old_ev, list):
        old_list = old_ev
    else:
        old_list = [old_ev]

    if new_ev is None:
        new_list = []
    elif isinstance(new_ev, list):
        new_list = new_ev
    else:
        new_list = [new_ev]

    merged = old_list[:]

    for ev in new_list:
        if ev and ev not in merged:
            merged.append(ev)

    return merged


def merge_properties(old_props, new_props):
    merged = dict(old_props or {})

    for key, value in (new_props or {}).items():
        if value is not None:
            merged[key] = value

    return merged

In [ ]:
def merge_kg_parts(kg_parts):
    entities_by_id = {}
    relations_seen = set()
    relations = []

    for part in kg_parts:
        for ent in part["entities"]:
            ent_id = ent["id"]

            if ent_id not in entities_by_id:
                ent_copy = dict(ent)
                ent_copy["evidence_text"] = merge_evidence([], ent.get("evidence_text"))
                entities_by_id[ent_id] = ent_copy

            else:
                entities_by_id[ent_id]["evidence_text"] = merge_evidence(
                    entities_by_id[ent_id].get("evidence_text"),
                    ent.get("evidence_text"),
                )

                entities_by_id[ent_id]["properties"] = merge_properties(
                    entities_by_id[ent_id].get("properties", {}),
                    ent.get("properties", {}),
                )

                old_conf = entities_by_id[ent_id].get("confidence", 0)
                new_conf = ent.get("confidence", 0)
                entities_by_id[ent_id]["confidence"] = max(old_conf, new_conf)

        for rel in part["relations"]:
            rel_key = (
                rel.get("source"),
                rel.get("target"),
                rel.get("type"),
                rel.get("source_field"),
                rel.get("target_field"),
            )

            if rel_key not in relations_seen:
                relations_seen.add(rel_key)

                rel_copy = dict(rel)
                rel_copy["evidence_text"] = merge_evidence([], rel.get("evidence_text"))

                if "confidence" not in rel_copy:
                    rel_copy["confidence"] = 1.0

                relations.append(rel_copy)

    return {
        "mode": "knowledge_graph",
        "metadata": {
            "model": MODELLM,
            "input_path": str(INPUT_PATH),
            "valid_parts": len(kg_parts),
            "errors": len(errors),
            "total_tokens": total_tokens,
            "estimated_cost": total_cost,
        },
        "knowledge_graph": {
            "entities": list(entities_by_id.values()),
            "relations": relations,
        },
    }

In [ ]:
contract_kg = merge_kg_parts(kg_parts)
save_json(contract_kg, KG_FULL_PATH)

print("Saved:", KG_FULL_PATH)
print("Total entities:", len(contract_kg["knowledge_graph"]["entities"]))
print("Total relations:", len(contract_kg["knowledge_graph"]["relations"]))

Saved: ../../infra/json/kg_extraction/bellicum_contract_kg.json
Total entities: 14
Total relations: 1


In [ ]:
print("Errors:", len(errors))

for err in errors[:5]:
    print("=" * 80)
    print("Clause:", err["clause_id"])
    print("Error:", err["error"])
    print("Text:", err["text"][:500])

Errors: 0
